# Creating a Mesh Model — documentation examples

Companion notebook to the **Creating a Mesh Model** documentation page
(`docs/source/mesh.rst`). One section per code snippet / figure; running the
whole notebook regenerates every figure on that page (light **and** dark
variants) into `docs/img/`.

Requires only the core `stellar-spice` install.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Figures are written straight into the documentation image directory with the
# exact filenames the .rst pages reference.
DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Render ``make_fig()`` twice — default (light) style and dark style — and
    save as <name>.png / <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")


## Basic Icosphere Creation

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import Blackbody

# Initialize a Blackbody model (for spectrum calculation)
bb = Blackbody()

# Create an icosphere model
m = IcosphereModel.construct(
    1000,  # Minimal number of vertices
    1.,     # Radius in solar radii
    1.,     # Mass in solar masses
    bb.to_parameters(),  # Parameters for the spectrum model
    bb.parameter_names   # Names of the parameters used to match between overridable parameters and mesh parameters
)

[spice] IcosphereModel constructed in 1.0 s


## Adding Rotation

In [3]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation

# Add rotation to the model
rm = add_rotation(
    m, # Model instance
    rotation_velocity=10., # Rotation velocity in km/s
    rotation_axis=jnp.array([0, 1, 0]) # Rotation axis (relative to the model center)
)  # Rotate around the y-axis (the default), which lies in the sky plane
   # of the default line of sight [0, 0, -1]

t = 1. # time in seconds

# To get the rotated mesh at a specific time:
rotated_mesh = evaluate_rotation(rm, t)  # t is the time in seconds

PHOEBE: passband "Bolometric:900-40000" has a newer version available.  Run phoebe.list_passband_online_history("Bolometric:900-40000") to get a list of available changes and phoebe.update_passband("Bolometric:900-40000") or phoebe.update_all_passbands() to update.


## Visualizing the Mesh

Figure: `rotated_mesh.png` / `rotated_mesh_dark.png`

In [4]:
from spice.plots import plot_3D

save_doc_fig("rotated_mesh",
             lambda: plot_3D(rotated_mesh, 'los_velocities')[0])

saved rotated_mesh.png / rotated_mesh_dark.png


## Adding Pulsation

Figure: `pulsated_mesh.png` / `pulsated_mesh_dark.png`

The pulsation period and the time passed to `evaluate_pulsations` are in days:
the conversion of pulsation velocities to km/s assumes solRad/day.

In [5]:
from spice.models.mesh_transform import add_pulsation, evaluate_pulsations

# Add pulsation to the model
mp = add_pulsation(
    m, # Model instance
    1, # m order
    1, # l degree
    100., # pulsation period
    jnp.array([[0.5, 0.]]) # Fourier series parameters
)

t = 40. # time in days, the same unit as the pulsation period

# To get the pulsated mesh at a specific time:
pulsated_mesh = evaluate_pulsations(mp, t)  # t is the time

[spice] Pulsations evaluated in 2.6 s


In [6]:
save_doc_fig("pulsated_mesh",
             lambda: plot_3D(pulsated_mesh, 'los_velocities')[0])

saved pulsated_mesh.png / pulsated_mesh_dark.png


## Travelling (prograde and retrograde) modes

A standing $m \neq 0$ mode fades in place; a travelling one keeps its amplitude and moves at $\omega/m$ in longitude. *Prograde* moves with the star's spin.

A travelling mode is $\mathrm{Re}[Y_l^m e^{\pm i\omega t}]$ per Fourier harmonic: upper sign prograde, lower sign retrograde (the code uses $e^{-i s \omega t}$ with $s=-1$ for prograde).

In [7]:
import numpy as np
from spice.models import IcosphereModel
from spice.models.mesh_transform import add_rotation, add_pulsation, evaluate_pulsations
from spice.models.utils import rotation_axis_frame

star = IcosphereModel.construct(5000, 2.0, 1.5, jnp.array([7500., 4.]), ["teff", "logg"])
star = add_rotation(star, 60.)       # add rotation first: modes are built about its axis

period = 0.2                         # days
radial = jnp.array([[[0.02, 0.]], [[0., 0.]], [[0., 0.]]])   # 2% radial amplitude

standing = add_pulsation(star, 4, 4, period, radial)
prograde = add_pulsation(star, 4, 4, period, radial, travelling="prograde")

snapshot = evaluate_pulsations(prograde, 0.05)   # t in days

[spice] IcosphereModel constructed in 1.3 s


[spice] Pulsations evaluated in 2.1 s


In [8]:
# longitude/latitude of each face about the rotation axis
xyz = np.asarray(star.d_centers) @ np.asarray(rotation_axis_frame(star.rotation_axis))
lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
lat = np.degrees(np.arcsin(xyz[:, 2] / np.linalg.norm(xyz, axis=1)))
r_hat = np.asarray(star.d_centers) / np.linalg.norm(np.asarray(star.d_centers), axis=1, keepdims=True)

def radial_displacement(model, t):
    offsets = np.asarray(evaluate_pulsations(model, t).center_pulsation_offsets)
    return np.einsum("ij,ij->i", offsets, r_hat) / float(star.radius)

times = period * np.array([0, 1, 2, 3]) / 16
maps = {name: [radial_displacement(model, t) for t in times]
        for name, model in [("Standing", standing), ("Prograde", prograde)]}

def snapshots_fig():
    fig, axes = plt.subplots(2, 4, figsize=(16, 5.6), sharex=True, sharey=True)
    for row, (name, frames) in zip(axes, maps.items()):
        for ax, t, values in zip(row, times, frames):
            sc = ax.scatter(lon, lat, c=values, s=6, cmap="RdBu_r", vmin=-0.02, vmax=0.02)
            ax.set_title(f"{name}, t = {t / period:.4g} P")
        row[0].set_ylabel("Latitude [deg]")
    for ax in axes[-1]:
        ax.set_xlabel("Longitude [deg]")
    fig.colorbar(sc, ax=axes, label=r"Radial displacement [$R_\star$]", shrink=0.9)
    return fig

save_doc_fig("travelling_mode_snapshots", snapshots_fig)

[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.0 s


saved travelling_mode_snapshots.png / travelling_mode_snapshots_dark.png


### Several modes with per-mode senses

In [9]:
from spice.models.mesh_transform import add_pulsations

modes = add_pulsations(
    star,
    m_orders=jnp.array([2, 3, 0]),
    l_degrees=jnp.array([2, 3, 2]),
    periods=jnp.array([0.21, 0.17, 0.25]),
    fourier_series_parameters=jnp.stack([radial, 0.5 * radial, 0.5 * radial]),
    travelling=["prograde", "retrograde", None],   # the m = 0 mode must stay standing
)
evaluated = evaluate_pulsations(modes, 0.1)
print(np.asarray(modes.pulsation_travelling)[np.asarray([2, 3, 0]) + int(modes.max_pulsation_mode) * np.asarray([2, 3, 2])])

[spice] Pulsations evaluated in 0.0 s


[-1.  1.  0.]


## Tilted Pulsation Axis

Figure: `pulsation_with_tilt.png` / `pulsation_with_tilt_dark.png`

In [10]:
untilted_m = add_pulsation(m, 0, 1, 2., jnp.array([[1e-1, 0.]]))
tilted_m = add_pulsation(m, 0, 1, 2., jnp.array([[1e-1, 0.]]),
                         pulsation_axes=jnp.array([1., 0., 0.]),
                         pulsation_angles=jnp.array([45.]))

untilted_pulsated_m = evaluate_pulsations(untilted_m, 0.5)
tilted_pulsated_m = evaluate_pulsations(tilted_m, 0.5)

def tilt_comparison_fig():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), subplot_kw={'projection': '3d'})
    plot_3D(untilted_pulsated_m, property='los_velocities', axes=(fig, ax1))
    ax1.set_title('Untilted Pulsation')
    plot_3D(tilted_pulsated_m, property='los_velocities', axes=(fig, ax2))
    ax2.set_title('Tilted Pulsation')
    fig.tight_layout()
    return fig

save_doc_fig("pulsation_with_tilt", tilt_comparison_fig)

[spice] Pulsations evaluated in 0.0 s


[spice] Pulsations evaluated in 0.3 s


saved pulsation_with_tilt.png / pulsation_with_tilt_dark.png


## Adding Temperature Spots — spherical harmonic

Figure: `temp_harmonic.png` / `temp_harmonic_dark.png`

In [11]:
from spice.models.spots import add_spherical_harmonic_spot

# Add a temperature pattern following Re[Y_4^4]
m_harm_spot = add_spherical_harmonic_spot(
    m, # Model instance
    4, # m order
    4, # l degree
    param_delta=2000, # amplitude multiplying Re[Y_l^m]; added on top of the existing value
    param_index=0 # index of the parameter in the parameters array
)

print('Teff range:', float(m_harm_spot.parameters[:, 0].min()), float(m_harm_spot.parameters[:, 0].max()))

save_doc_fig("temp_harmonic",
             lambda: plot_3D(m_harm_spot, property=0, property_label='Temperature [K]')[0])

Teff range: 4911.557921530008 6648.070763873042


saved temp_harmonic.png / temp_harmonic_dark.png


## Tilted spherical-harmonic spot

Figure: `tilted_temperature_spot.png` / `tilted_temperature_spot_dark.png`

In [12]:
m_harm_spot_tilted = add_spherical_harmonic_spot(
    m, # Model instance
    4, # m order
    4, # l degree
    param_delta=2000, # amplitude multiplying Re[Y_l^m]
    param_index=0, # index of the parameter in the parameters array
    tilt_axis=jnp.array([1., 0., 0.]),  # perpendicular to the default +y rotation axis
    tilt_angle=45.  # tilt angle in degrees
)

save_doc_fig("tilted_temperature_spot",
             lambda: plot_3D(m_harm_spot_tilted, property=0, property_label='Temperature [K]')[0])

saved tilted_temperature_spot.png / tilted_temperature_spot_dark.png


## Circular spot

Figure: `temp_spot.png` / `temp_spot_dark.png`

In [13]:
from spice.models.spots import add_spot
from spice.models.utils import lat_to_theta, lon_to_phi

m_spot = add_spot(
    mesh=m, # Model instance
    spot_center_theta=lat_to_theta(30.), # colatitude from the rotation axis, in radians
    spot_center_phi=lon_to_phi(-90.), # longitude about the rotation axis, in radians (-90 deg faces the default observer)
    spot_radius=50., # spot radius in degrees
    parameter_delta=1000, # difference in the parameter value between the spot and the background
    parameter_index=0, # index of the parameter in the parameters array
    smoothness=0.1 # smoothness of the spot edges
)

save_doc_fig("temp_spot",
             lambda: plot_3D(m_spot, property=0, property_label='Temperature [K]')[0])

saved temp_spot.png / temp_spot_dark.png


## Two spherical-harmonic modes

Figures: `temp_2_2_3_3.png` (+`_dark`) and the side-by-side
`temp_2_2_3_3_comparison.png` (+`_dark`)

In [14]:
from spice.models.spots import add_spherical_harmonic_spots

m_harm_spots = add_spherical_harmonic_spots(
    mesh=m,
    m_orders=jnp.array([2, 3.]),
    n_degrees=jnp.array([2, 3.]),
    param_deltas=jnp.array([1000, 1000.]),
    param_indices=jnp.array([0, 0])
)

save_doc_fig("temp_2_2_3_3",
             lambda: plot_3D(m_harm_spots, property=0, property_label='Temperature [K]')[0])

saved temp_2_2_3_3.png / temp_2_2_3_3_dark.png


In [15]:
m_22 = add_spherical_harmonic_spot(m, 2, 2, param_delta=1000, param_index=0)
m_33 = add_spherical_harmonic_spot(m, 3, 3, param_delta=1000, param_index=0)

def harmonic_comparison_fig():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), subplot_kw={'projection': '3d'})
    plot_3D(m_22, property=0, property_label='Temperature [K]', axes=(fig, ax1))
    ax1.set_title('m=2, l=2')
    plot_3D(m_33, property=0, property_label='Temperature [K]', axes=(fig, ax2))
    ax2.set_title('m=3, l=3')
    fig.tight_layout()
    return fig

save_doc_fig("temp_2_2_3_3_comparison", harmonic_comparison_fig)

saved temp_2_2_3_3_comparison.png / temp_2_2_3_3_comparison_dark.png


## Two circular spots

Figure: `temp_two_spots.png` / `temp_two_spots_dark.png`

In [16]:
from spice.models.spots import add_spots
from spice.models.utils import lat_to_theta, lon_to_phi

m_spots = add_spots(
    mesh=m, # Model instance
    spot_center_thetas=lat_to_theta(jnp.array([35., -20.])), # colatitudes from the rotation axis, in radians
    spot_center_phis=lon_to_phi(jnp.array([-60., -120.])), # longitudes about the rotation axis, in radians
    spot_radii=jnp.array([20., 20.]), # spot radius in degrees
    parameter_deltas=jnp.array([1000., 1000.]), # difference in the parameter value between the spot and the background
    parameter_indices=jnp.array([0, 0]), # index of the parameter in the parameters array
    smoothness=jnp.array([0.5, 0.5]) # smoothness of the spot edges
)

save_doc_fig("temp_two_spots",
             lambda: plot_3D(m_spots, property=0, property_label='Temperature [K]')[0])

saved temp_two_spots.png / temp_two_spots_dark.png
